In [4]:
import pandas as pd

df = pd.read_csv("../data/contract_evaluation_dataset_humera_khan.csv")

# Verify required columns
required_cols = [
    "contract_text",
    "expected_apr",
    "expected_term",
    "expected_payment",
    "expected_penalty"
]

df.head()

,contract_id,contract_text,expected_apr,expected_term,expected_payment,expected_penalty
0,1,The borrower agrees to an APR of 12%. The loan...,12.0,24.0,4500.0,late fee 200
1,2,This agreement offers a loan at an APR of 10.5...,10.5,36.0,3200.0,NaN
2,3,A personal loan with a term of 18 months. Mont...,NaN,18.0,5200.0,NaN
3,4,The customer will pay 4100 every month for 12 ...,14.0,12.0,4100.0,late fee 150
4,5,Loan sanctioned with APR of 9%. Penalty of 300...,9.0,NaN,NaN,missed payment fee 300


In [5]:
expected_output_format = {
    "apr": None,
    "term_months": None,
    "monthly_payment": None,
    "penalty_clause": None
}


In [6]:
PROMPT_TEMPLATE = """
You are an information extraction assistant.

Extract ONLY the following fields from the contract text:
1. APR
2. Loan term in months
3. Monthly payment amount
4. Penalty clause

Rules:
- Return ONLY valid JSON
- If a value is NOT explicitly mentioned, return null
- Do NOT infer or assume any values
- Match exactly what is written in the contract
- Do NOT add explanations

Output format:
{
  "apr": null,
  "term_months": null,
  "monthly_payment": null,
  "penalty_clause": null
}

Contract Text:
\"\"\"{contract_text}\"\"\"
"""


In [11]:
import json
import re

def extract_sla(contract_text):
    """
    Baseline LLM-style extraction function.
    Does NOT infer missing values.
    Returns None for missing fields.
    """

    output = {
        "apr": None,
        "term_months": None,
        "monthly_payment": None,
        "penalty_clause": None
    }

    # Simple pattern checks (baseline logic)
    apr_match = re.search(r"APR\s*(of|is)?\s*([0-9]+(\.[0-9]+)?)%", contract_text)
    if apr_match:
        output["apr"] = apr_match.group(2)

    term_match = re.search(r"([0-9]+)\s*months", contract_text)
    if term_match:
        output["term_months"] = term_match.group(1)

    payment_match = re.search(r"monthly (payment|installment).*?([0-9]+)", contract_text, re.IGNORECASE)
    if payment_match:
        output["monthly_payment"] = payment_match.group(2)

    penalty_match = re.search(
        r"(late fee|late payment fee|early termination fee|missed payment fee|penalty).*?([0-9]+)",
        contract_text,
        re.IGNORECASE
    )
    if penalty_match:
        output["penalty_clause"] = penalty_match.group(0)

    return output


In [12]:
sample_df = df.sample(5, random_state=42)

results = []

for _, row in sample_df.iterrows():
    extracted = extract_sla(row["contract_text"])

    results.append({
        "contract_text": row["contract_text"],
        "expected_apr": row["expected_apr"],
        "expected_term": row["expected_term"],
        "expected_payment": row["expected_payment"],
        "expected_penalty": row["expected_penalty"],
        "llm_apr": extracted["apr"],
        "llm_term": extracted["term_months"],
        "llm_payment": extracted["monthly_payment"],
        "llm_penalty": extracted["penalty_clause"]
    })

result_df = pd.DataFrame(results)
result_df

,contract_text,expected_apr,expected_term,expected_payment,expected_penalty,llm_apr,llm_term,llm_payment,llm_penalty
0,APR is 13%. Early termination penalty of 400 a...,13.0,NaN,NaN,early termination fee 400,13,None,None,penalty of 400
1,The agreement specifies APR of 7.5% and a mont...,7.5,36.0,2800.0,NaN,7.5,36,2800,None
2,The loan will run for 72 months. Penalty appli...,NaN,72.0,NaN,NaN,None,72,None,None
3,APR is mentioned as 16%. Penalty of 200 applie...,16.0,NaN,NaN,late fee 200,None,None,None,Penalty of 200
4,Loan agreement with APR 13.5% and term of 36 m...,13.5,36.0,4200.0,late fee 250,13.5,36,4200,penalty is 250


In [13]:
def match(a, b):
    if pd.isna(a) and pd.isna(b):
        return 1
    return int(str(a).strip() == str(b).strip())

result_df["apr_match"] = result_df.apply(lambda x: match(x["expected_apr"], x["llm_apr"]), axis=1)
result_df["term_match"] = result_df.apply(lambda x: match(x["expected_term"], x["llm_term"]), axis=1)
result_df["payment_match"] = result_df.apply(lambda x: match(x["expected_payment"], x["llm_payment"]), axis=1)
result_df["penalty_match"] = result_df.apply(lambda x: match(x["expected_penalty"], x["llm_penalty"]), axis=1)

result_df


,contract_text,expected_apr,expected_term,expected_payment,expected_penalty,llm_apr,llm_term,llm_payment,llm_penalty,apr_match,term_match,payment_match,penalty_match
0,APR is 13%. Early termination penalty of 400 a...,13.0,NaN,NaN,early termination fee 400,13,None,None,penalty of 400,0,1,1,0
1,The agreement specifies APR of 7.5% and a mont...,7.5,36.0,2800.0,NaN,7.5,36,2800,None,1,0,0,1
2,The loan will run for 72 months. Penalty appli...,NaN,72.0,NaN,NaN,None,72,None,None,1,0,1,1
3,APR is mentioned as 16%. Penalty of 200 applie...,16.0,NaN,NaN,late fee 200,None,None,None,Penalty of 200,0,1,1,0
4,Loan agreement with APR 13.5% and term of 36 m...,13.5,36.0,4200.0,late fee 250,13.5,36,4200,penalty is 250,1,0,0,0
